### <h1 align="center">**Sigmoid CURVE FIT function algorithm for pole_to_pole distance for individual fit data**</h1> 

This script fits an exponential curve to the **individual** of each cell type of the *C. elegans* embryo in batch process. This curve uses the sigmoid fitting function where 
\begin{equation*}
L = a + \big(\frac{b}{1 + e^\frac{-(t-t0)}{\tau}}\big) 
\end{equation*}

The goal for fitting a mathematical function to the mean value is to determine the **Initial pole-to-pole length**, **Final pole-to-pole length**, **Elongation rate** and **Metaphase_length** at the required time point. 

`INPUT FILES` 

The csv files of each cell type containing the pole-to-pole distance data of different experiments (n-value). 

The table below shows the table structure of each .csv file. The headers of each table should show the time series measurement value from each observation which starts with prefix, **Exp**, and other statistical values from the observations.

|Exp00 | Exp01 | Exp02 | Exp03 | Exp04 | Exp05 | mean | std | n | SE | time |
| :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----: | :-----:| :-----: | :-----: |
| `num` | `num` | `num` | `num` | `num` | `num` | `num` | `num` | `num`| `num` | `num` |
| `...` | `...` | `...` | `...` | `...` | `...` | `...` | `...` | `...`| `...` | `...` |

`OUTPUT FILES` 

* **FIRST_GROUP_OUTPUT_FILES**: The *.png* files of the fitted plot of each cell type. 

* **SECOND_GROUP_OUTPUT_FILE**: A *.csv* file containing the **Initial pole_to_pole length (µm)**, the **Final pole_to_pole length (µm)**, the **Elongation rate (µm/minute)** and the **Metaphase_length** of each of the cell type. 

In [1]:
# library packages
import os
import warnings
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from numpy import exp, linspace, random, arange
from scipy.optimize import curve_fit, least_squares

In [2]:
# input folder
folder = r'D:\data\Analysis Data\python_analysis\input'

# output folder
save_files = r'D:\data\Analysis Data\python_analysis\output'

In [3]:
# new DataFrame to append the new generated table 
fit_Result = pd.DataFrame()

# read out individual files and compute for different operations as defined in the for loop
for file in os.scandir(folder):
    df = pd.read_csv(file)
    
    # create a new DataFrame 
    Exp_Column = df.loc[:, df.columns.str.startswith('Exp')]
    mean_column = df.loc[:, df.columns.str.startswith('mean')]
    time_column = df.loc[:, df.columns.str.startswith('time')]
    
    # define the start and end time points to compute
    x_start = np.where(time_column >= -103)[0][0]
    x_end = np.where((time_column <= 206) & (time_column == 206))[0][0] + 1
    
    # compute the time frames for column values
    new_x = time_column[x_start:x_end]
    new_exp = Exp_Column[x_start:x_end]
    new_mean = mean_column[x_start:x_end]
    
    # create a new DataFrame
    df_Table = pd.concat([new_x, new_exp, new_mean], axis=1)
    
    '''
    drop all the rows were n-value is less than 3 the mean and the time 
    columns are included, ie, the number of rows to be computed should be >= 3
    '''
    newTable = df_Table.dropna(thresh=3)   
    
    # define a logistics function to fit for a sigmoid curved data
    '''
    t = independent variable [time or position]
    a = the lower asymptote of the sigmoid curve. It determines the minimum  value that the curve approaches as 't' approaches negative infinity.
    b = the upper asymptote of the sigmoid curve. It represents the maximum value that the curve approaches as 't' goes to positive infinity.
    t0 = x-value of the sigmoid's midpoint (the inflection point)
    k = rate of growth (controlling the steepness of the curve around the inflection point)
    '''
    def sigmoid(t, a, b, t0, k):
        L = a + (b / (1+(np.exp(-(t-t0)/k))))
        return L
    
    warnings.filterwarnings("ignore") # ignore warning 
    plt.figure(figsize=(5,5)) # define the plot dimension
    
    # iterate over every column in each DataFrame and plot
    for i_col in newTable.columns[1:-1]:
        try:
            NaN_table = newTable[['time', i_col]]        
            plot_table = NaN_table.dropna()
            
            # x and y variables
            x = plot_table['time']
            y = plot_table[i_col]
            
            # compute the initial guess
            initial_guess = [max(y)-min(y), 5, 0.05, min(y)]
            
            # define the boundary parameter to ensure non-negative 'a' value
            parameter_bounds = ([0, -np.inf, -np.inf, 0], [np.inf, np.inf, np.inf, np.inf])

            # summarize the parameter
            popt, pcov = curve_fit(sigmoid, x, y, initial_guess, bounds=parameter_bounds, maxfev=10000)

            ''' 
            Define a sequence of inputs between the smallest and largest known inputs and define the fit. 
            Let the maximum input assume the maximum values of x. 
            '''
            x_fit = np.arange(x.min()-0.01, x.max()+0.01, 0.01)
            y_fit = sigmoid(x_fit, *popt)

            # primary plot
            ax1 = sns.scatterplot(x=x, y=y, alpha=0.2, label=i_col)

            # fit on each plot 
            sns.lineplot(x=x_fit, y=y_fit, alpha = 1, ax=ax1)

            # add the desired features on the plot
            ax1.set_xlabel('time [s]', fontsize= 18)
            ax1.set_ylabel('distance [μm]', fontsize= 18)
            plot_title = (file.name).split('.')[0]
            ax1.axes.set_title(plot_title, fontsize= 20, fontweight='bold')  
            ax1.set(ylim=(0, 30), xlim=(0, 200))
            ax1.set(xlim=(-120, 220))

            # add a vertical line at time 0 sec to indicate anaphase onset
            ax1.vlines(x=0, ymin=0, ymax=30, linestyle='solid', color='red')

            # plot_files = os.path.join(save_files)
            plotfile = (file.name).split('.')[0] + '.png'
            plt.savefig(os.path.join(save_files, plotfile), dpi=300)
            
            # assign variables to the fit parameter output
            a = popt[0]
            b = popt[1]
            t0 = popt[2]
            k = popt[3]

            initial_length = a
            final_length = a + b
            elongation_rate = (b/(4*k))*60 # multiple by 60 to give the final value in µm/minute
            
            index_zero = np.argmin(np.abs(x_fit - 0)) # define the value of x_fit at time == 0
            metaphase_P_P_length = y_fit[index_zero] # define the value of y_fit at index_zero

            # create a new dictionary for each parameter and add to the DataFrame fit_Result
            parameters = {'Initial pole_pole length (µm)': initial_length, 
                          'Final pole_pole length (µm)': final_length, 
                          'Elongation rate (µm/min)': elongation_rate,
                          'Metaphase_length (µm)': metaphase_P_P_length}

            parameters_df = pd.DataFrame.from_dict(parameters, orient='index', columns=[file.name.split('.')[0]])
            fit_Result = pd.concat([fit_Result, parameters_df], axis=1)
        
        except Exception:
            pass

# close all plots open windows
plt.close('all')
        
# transpose the table
fit_Result_transpose = (fit_Result).T

# assign header to the index column
fit_Result_transpose.index.names = ['Cells']

# save the table, fit_Result, to a csv file
fit_Result_transpose.to_csv(os.path.join(save_files, 'Fit_Result_pole_pole.csv'), encoding='utf-8')